In [1]:
import os
import numpy as np
import pandas as pd
import random
import torch
from torch import nn
from glob import glob
from tqdm.notebook import tqdm
import pickle

In [2]:
cfg_size=1000
PATH = '../data/npz_all/npz/layout'
OUT = f'../data/layout_converted_z2_{cfg_size}'
#cfg_size=128


os.makedirs(OUT, exist_ok=True)

data_graphs = {}
for fname in tqdm(list(glob(os.path.join(PATH,'**/*.npz'), recursive=True))):
    idx = '.'.join(fname.replace(PATH,'').split('.')[:-1])[1:]
    train = 'train' in fname
    data = np.load(fname)
    
    data_graphs[idx] = {
        'edge_index':data['edge_index'],
        'node_feat':data['node_feat'],
        'node_opcode':data['node_opcode'],
        'node_config_ids':data['node_config_ids'],
    }
    
    configs = (data['node_config_feat']).astype(np.int8)
    runtime = data['config_runtime']
    runtime_norm = (runtime - runtime.mean())/(runtime.std()+1e-4)
    runtime_norm = runtime_norm.astype(np.float32)
    L = runtime.shape[0]
    
    out = os.path.join(OUT,idx)
    os.makedirs(out, exist_ok=True)
    for i,s in enumerate(range(0,L,cfg_size)):
        f = min(s + cfg_size,L)
        if train and f - s < cfg_size//2: continue
        np.savez_compressed(os.path.join(out, f'{i}.npz'), 
                    node_config_feat = configs[s:f].copy(), 
                    config_runtime = runtime[s:f].copy(),
                    config_runtime_norm = runtime_norm[s:f].copy())
        
with open(os.path.join(OUT,'graphs.pickle'), 'wb') as f:
    pickle.dump(data_graphs, f, protocol=pickle.HIGHEST_PROTOCOL)

  0%|          | 0/639 [00:00<?, ?it/s]